In [1]:
# Import libaries and modules
import pandas as pd
import numpy as np
import sys
import os

print("[i] INFO - Program has been initialised")

[i] INFO - Program has been initialised


In [2]:
# Pre-define variables
rs = 1
folder, input, output = 'datasets', 'dataset.csv', 'dataset_proc.csv'

keep_triplets = [
    (53, "udp", "dns"), (1883, "tcp", "mqtt"), (80, "tcp", "http"), (443, "tcp", "ssl")
]

In [3]:
def load_dataset():
    '''
    Load the dataset from the specified path.

    Parameters:
        None

    Returns:
        df
    '''

    path = os.path.join(folder, input)
    print(f"[i] INFO - Loading '{input}' dataset...")
    try:
        df = pd.read_csv(path, low_memory=False)
        print(f"[~] DEBUG - Loaded dataset with shape: ({df.shape[0]} x {df.shape[1]})")
    except FileNotFoundError:
        print(f"[!] ERROR - No file found at '{path}'")
        input();sys.exit(1)
    except PermissionError:
        print(f"[!] ERROR - No permission to read '{path}'")
        input();sys.exit(1)
    except UnicodeDecodeError:
        print(f"[!] ERROR - Unable to decode '{path}'")
        input();sys.exit(1)
    except pd.errors.EmptyDataError:
        print(f"[!] ERROR - No data found in '{path}'")
        input();sys.exit(1)
    except pd.errors.ParserError:
        print(f"[!] ERROR - Unable to parse data in '{path}'")
        input();sys.exit(1)

    return df

In [4]:
def save_dataframe(df_proc):
    '''
    Saves the dataframe to the specified path.

    Parameters:
        df_sample

    Returns:
        None
    '''

    path = os.path.join(folder, output)
    print("[i] INFO - Saving dataset...")

    try:
        df_proc.to_csv(path, index=False)
        print(f"[~] DEBUG - Saved dataset with shape: ({df_proc.shape[0]} x {df_proc.shape[1]})")
    except FileNotFoundError:
        print(f"[!] ERROR - No directory found at '{path}'")
        input();sys.exit(1)
    except PermissionError:
        print(f"[!] ERROR - No permission to write to '{path}'")
        input();sys.exit(1)

    return

In [ ]:
def process_dataframe(df):
    '''
    Process the dataframe by dropping invalid values, removing duplicates and etc.

    Parameters:
        df

    Returns:
        df
    '''

    print(f"[i] INFO - Processing '{input}' dataframe...")

    # Replace - values with NaN
    df = df.replace(["-"], np.nan)

    # Remove NaN, NaT and None values
    df = df.dropna()

    # Remove rows containing duplicate data
    df = df.drop_duplicates()

    # Remove columns containing values that cause data leakage globally
    df = df.drop(columns=["fwd_pkts_per_sec", "bwd_pkts_per_sec", "flow_pkts_per_sec", "fwd_iat.min", "fwd_iat.max", "fwd_iat.tot", "fwd_iat.avg", "fwd_iat.std", "bwd_iat.min", "bwd_iat.max"])

    # Remove columns containing constant or almost constant values globally
    df = df.drop(columns=["fwd_URG_flag_count", "bwd_URG_flag_count", "flow_CWR_flag_count", "flow_ECE_flag_count"])
    
    # Remove columns containing non-informative data
    df = df.drop(columns=["id.orig_p"])

    # Remove low-frequency combinations
    df = df[df[["id.resp_p", "proto", "service"]].apply(tuple, axis=1).isin(keep_triplets)]

    # Remove protocol and service column
    df = df.drop(columns=["proto", "service"])

    return df

In [6]:
def main():
    '''
    Process the dataset.

    Parameters:
        None

    Returns:
        None
    '''

    # Load the dataset
    df = load_dataset()

    # Process the dataframe
    df_proc = process_dataframe(df)

    # Save the dataframe to a .csv file
    save_dataframe(df_proc)

    print("[i] INFO - Program has finished execution")

if __name__ == "__main__":
    main()

[i] INFO - Loading 'dataset.csv' dataset...
[~] DEBUG - Loaded dataset with shape: (123117 x 51)
[i] INFO - Processing 'dataset.csv' dataframe...
[i] INFO - Saving dataset...
[~] DEBUG - Saved dataset with shape: (16954 x 34)
[i] INFO - Program has finished execution
